# Lap Coach Debug Notebook

Quick exploration of MCAP lap data and processing outputs.

In [ ]:
import numpy as np
from backend.processing.parser import LapDataParser, _arc_length
from backend.processing.brake_analysis import detect_brake_events, match_brake_events, print_brake_recommendations
from backend.processing.gas_analysis import (
    detect_throttle_applications,
    detect_throttle_plateaus,
    match_throttle_applications,
    match_throttle_plateaus,
    print_gas_recommendations,
)
from backend.processing.steering_analysis import detect_corners, match_corners, print_steering_recommendations


In [ ]:
# Load laps
fast_path = "data/hackathon/hackathon_fast_laps.mcap"
good_path = "data/hackathon/hackathon_good_lap.mcap"

fast_states = LapDataParser(fast_path).get_lap_data()
good_states = LapDataParser(good_path).get_lap_data()

print(f"Fast lap samples: {len(fast_states)}")
print(f"Good lap samples: {len(good_states)}")


In [ ]:
# Build arc-length (metres along track)
fast_arc = _arc_length(fast_states)
good_arc = _arc_length(good_states)

print(f"Fast lap length: {fast_arc[-1]:.1f} m")
print(f"Good lap length: {good_arc[-1]:.1f} m")


In [ ]:
# Raw channel quick look
def describe_channel(states, name):
    arr = np.array([getattr(s, name) for s in states], dtype=float)
    return dict(min=float(arr.min()), max=float(arr.max()), mean=float(arr.mean()))

for ch in ["steering", "brake", "gas", "speed"]:
    print(ch, "fast", describe_channel(fast_states, ch))
    print(ch, "good", describe_channel(good_states, ch))


In [ ]:
# Brake analysis
fast_brakes = detect_brake_events(fast_states, fast_arc)
good_brakes = detect_brake_events(good_states, good_arc)
print(f"Fast brake zones: {len(fast_brakes)}")
print(f"Good brake zones: {len(good_brakes)}")

brake_matches = match_brake_events(fast_brakes, good_brakes)
print_brake_recommendations(brake_matches)


In [ ]:
# Gas analysis
fast_apps = detect_throttle_applications(fast_states, fast_arc)
good_apps = detect_throttle_applications(good_states, good_arc)

fast_plats = detect_throttle_plateaus(fast_states, fast_arc)
good_plats = detect_throttle_plateaus(good_states, good_arc)

app_matches = match_throttle_applications(fast_apps, good_apps)
plat_matches = match_throttle_plateaus(fast_plats, good_plats)

print_gas_recommendations(app_matches, plat_matches)


In [ ]:
# Steering analysis
fast_corners = detect_corners(fast_states, fast_arc)
good_corners = detect_corners(good_states, good_arc)
print(f"Fast corners: {len(fast_corners)}")
print(f"Good corners: {len(good_corners)}")

fast_steer = np.array([s.steering for s in fast_states], dtype=float)
good_steer = np.array([s.steering for s in good_states], dtype=float)

corner_matches = match_corners(
    fast_corners, good_corners,
    fast_arc, good_arc,
    fast_steer, good_steer,
)
print_steering_recommendations(corner_matches)
